In [3]:
import pandas as pd
from langchain_community.document_loaders import DataFrameLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.llms import Ollama
from langchain_core.prompts import PromptTemplate
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
import os

# ==========================================
# 1. CONFIGURAÇÕES INICIAIS
# ==========================================
MODEL_NAME = "gemma3:12b"
EMBEDDING_MODEL = "all-MiniLM-L6-v2" # Modelo leve e eficiente para criar os vetores em português/inglês

print("Iniciando o sistema RAG para Classificação de CID...")

# Inicializa o LLM via Ollama
llm = Ollama(model=MODEL_NAME, temperature=0.1) # Temperatura baixa para respostas mais determinísticas/precisas

# ==========================================
# 2. PREPARAÇÃO DA BASE DE CONHECIMENTO (RAG)
# ==========================================
print("Carregando bases de conhecimento...")
# Substitua pelo caminho correto se os arquivos não estiverem na mesma pasta
df_snomed = pd.read_csv('SNOMED_to_CID_2.csv')
df_base_cid = pd.read_csv('Base CID_Classificação CR.csv')

# --- ATENÇÃO: Ajuste os nomes das colunas abaixo conforme os seus arquivos ---
# Vamos criar uma coluna de "Texto_Busca" que unifica as informações para o modelo ler
df_snomed['Texto_Busca'] = "Mapeamento SNOMED-CID. SNOMED: " + df_snomed['SNOMED'].astype(str) + " | CID Correspondente: " + df_snomed['CID'].astype(str)
df_base_cid['Texto_Busca'] = "Regra de Classificação CID. Topografia: " + df_base_cid['TOPOGRAFIA'].astype(str) + " | Morfologia: " + df_base_cid['MORFOLOGIA'].astype(str) + " | Procedimento: " + df_base_cid['PROCEDIMENTO'].astype(str) + " | CID Resultante: " + df_base_cid['CID'].astype(str)

# Combina as duas bases de conhecimento
df_knowledge = pd.concat([df_snomed[['Texto_Busca']], df_base_cid[['Texto_Busca']]], ignore_index=True)

# Converte o DataFrame para documentos do LangChain
loader = DataFrameLoader(df_knowledge, page_content_column="Texto_Busca")
documents = loader.load()

# Cria os Embeddings e o Vector Store (Chroma)
print("Criando banco de dados vetorial (isso pode levar alguns instantes na primeira vez)...")
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
vectorstore = Chroma.from_documents(documents, embeddings, collection_name="cid_knowledge")
retriever = vectorstore.as_retriever(search_kwargs={"k": 5}) # Busca os 5 documentos mais relevantes

# ==========================================
# 3. CRIAÇÃO DO PROMPT E DA CHAIN (SINTAXE NOVA LCEL)
# ==========================================
prompt_template = """Você é um especialista em codificação médica (Auditor de Contas Médicas).
Sua tarefa é determinar o código CID correto do paciente baseando-se estritamente no contexto fornecido (regras de SNOMED, Topografia, Morfologia e Procedimento).

Contexto extraído da base de dados:
{context}

Informações do Paciente:
{input}

Instruções:
1. Analise as informações do paciente.
2. Busque no contexto a regra ou o mapeamento exato que se aplica.
3. Retorne APENAS o código CID final (ex: C50.9, Z00.0). Se houver uma explicação muito breve, coloque após o código, mas o código deve vir primeiro.
4. Se o contexto não contiver informações suficientes para determinar o CID com segurança, responda "CID_NÃO_ENCONTRADO". Não invente um código.

Resposta (Apenas o Código CID):"""

# Cria o prompt
PROMPT = PromptTemplate.from_template(prompt_template)

# Cria a chain de documentos (que junta os textos recuperados)
document_chain = create_stuff_documents_chain(llm, PROMPT)

# Cria a chain de recuperação final (que conecta o retriever ao LLM)
qa_chain = create_retrieval_chain(retriever, document_chain)
# ==========================================
# 4. PROCESSAMENTO DO ARQUIVO DE PACIENTES
# ==========================================
arquivo_entrada = "Resultado_SNOMED_CID_Parcial - Copia.csv"
arquivo_saida = "Resultado_Final_com_CID_Predito.csv"

print(f"Processando arquivo de pacientes: {arquivo_entrada}...")
df_pacientes = pd.read_csv(arquivo_entrada)

# Lista para armazenar as previsões
cids_preditos = []

# Itera sobre cada paciente no arquivo
for index, row in df_pacientes.iterrows():
    # --- ATENÇÃO: Ajuste os nomes das colunas de acordo com o seu arquivo de entrada ---
    diagnostico = row.get('DIAGNÓSTICO', 'Não informado')
    snomed = row.get('SNOMED', 'Não informado')
    topografia = row.get('TOPOGRAFIA', 'Não informado')
    procedimento = row.get('PROCEDIMENTO', 'Não informado')
    
    # Monta a "pergunta" que será enviada ao RAG
    query_paciente = f"Diagnóstico/Morfologia: {diagnostico} (SNOMED: {snomed}) | Topografia Anatômica: {topografia} | Procedimento: {procedimento}"
    
    print(f"[{index+1}/{len(df_pacientes)}] Analisando paciente...")
    
    try:
        # Consulta o RAG usando a nova chave "input"
        resposta = qa_chain.invoke({"input": query_paciente})
        
        # Na nova arquitetura, a resposta final costuma vir na chave 'answer'
        cid_resultado = resposta['answer'].strip() 
        print(f"  -> Resultado: {cid_resultado}")
    except Exception as e:
        print(f"  -> Erro ao processar: {e}")
        cid_resultado = "ERRO_SISTEMA"
        
    cids_preditos.append(cid_resultado)

# Adiciona os resultados ao DataFrame e salva o novo CSV
df_pacientes['CID_PREDITO_RAG'] = cids_preditos
df_pacientes.to_csv(arquivo_saida, index=False, encoding='utf-8')

print(f"Processo concluído com sucesso! Arquivo salvo em: {arquivo_saida}")

ModuleNotFoundError: No module named 'langchain.chains'